In [1]:
import sys
import os

from transformers import AutoTokenizer, AutoModelForCausalLM
import datasets
from functools import partial

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.dataset_tokenization import process_data

In [2]:
model_path = "MathBite/self_corrective_llama_3.1_8B_untrained"
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

In [3]:
# SPECIAL_INSTRUCTION = "\nAs you write your answer, you can correct yourself using these tools: Use <DEL_S> to remove the entire sentence before this token, and <DEL_A> to scrap everything you've written and start again."
# SPECIAL_INSTRUCTION = "\nIf you realize you have made a mistake, you must use one of the following tools to correct it: Use <DEL_S> to retract the entire sentence immediately preceding this token. Use <DEL_A> to retract your entire response and start over."
SPECIAL_INSTRUCTION = "\nIf you realize you have made a mistake, you must use one of the following correction tokens to fix it: Use <DEL_S> to delete the entire sentence immediately preceding this token. Use <DEL_A> to delete your entire response and start over."

INSERTION_MARKER = "<|eot_id|><|start_header_id|>user<|end_header_id|>"
DELETION_MARKERS = ["<DEL_S>", "<DEL_A>"]
DELETION_TOKEN_IDS = tokenizer.convert_tokens_to_ids(DELETION_MARKERS)

In [4]:
# data_path = "../../dataset/final_train_dataset.json"
data_path = "../../dataset/s1_train_dataset.json"
dataset = datasets.load_dataset("json", data_files=data_path)

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'incorrect_response', 'errors', 'hallucinated_text', 'correct_response', 'additional_info'],
        num_rows: 35019
    })
})

In [6]:
for i in range(100):
    # i = 0
    sample = dataset["train"][i]

    res = process_data(sample, tokenizer, SPECIAL_INSTRUCTION, INSERTION_MARKER, DELETION_TOKEN_IDS[0], DELETION_TOKEN_IDS[1])

    hall_text_idx = [i for i, label in enumerate(res["hallucination_labels"]) if label == 1]
    hall_text = [res["input_ids"][i] for i in hall_text_idx]
    token_labels = [res["labels"][i] for i in range(len(res["labels"])) if res["labels"][i] != -100]

    print(str(i) + " Full response:\n" + sample["correct_response"])
    print("================================================")
    print("Hallucinated text:")
    for error in sample["hallucinated_text"]:
        print(error)
    print("================================================")
    print("Hallucinated text from tokenized response:")
    print(tokenizer.decode(hall_text))
    print("================================================")
    print("Tokens model will learn from:")
    print(tokenizer.decode(token_labels))
    print("--------------------------------\n\n")


0 Full response:
Parkwood Topshop Athletic Ltd.<DEL_A> activewear
Hallucinated text:
Parkwood Topshop Athletic Ltd.
Hallucinated text from tokenized response:
Parkwood Topshop Athletic Ltd.<DEL_A>
Tokens model will learn from:
<DEL_A> activewear<|eot_id|>
--------------------------------


1 Full response:
Changes such as women wanting to be priests, priests wanting to get married, bishops becoming regional popes, and theologians claiming absolute teaching authority.<DEL_A> controversial
Hallucinated text:
Changes such as women wanting to be priests, priests wanting to get married, bishops becoming regional popes, and theologians claiming absolute teaching authority.
Hallucinated text from tokenized response:
Changes such as women wanting to be priests, priests wanting to get married, bishops becoming regional popes, and theologians claiming absolute teaching authority.<DEL_A>
Tokens model will learn from:
<DEL_A> controversial<|eot_id|>
--------------------------------


2 Full respon

In [7]:
# SPECIAL_INSTRUCTION = "\nAs you write your answer, you can correct yourself using these tools: Use <DEL_S> to remove the entire sentence before this token, and <DEL_A> to scrap everything you've written and start again."
# SPECIAL_INSTRUCTION = "\nIf you realize you have made a mistake, you must use one of the following tools to correct it: Use <DEL_S> to retract the entire sentence immediately preceding this token. Use <DEL_A> to retract your entire response and start over."

SPECIAL_INSTRUCTION = "\nIf you realize you have made a mistake, you must use one of the following correction tokens to fix it: Use <DEL_S> to delete the entire sentence immediately preceding this token. Use <DEL_A> to delete your entire response and start over."

INSERTION_MARKER = "<|eot_id|><|start_header_id|>user<|end_header_id|>"
DELETION_MARKERS = ["<DEL_S>", "<DEL_A>"]
DELETION_TOKEN_IDS = tokenizer.convert_tokens_to_ids(DELETION_MARKERS)

mapper = partial(
    process_data,
    tokenizer=tokenizer,
    special_instruction=SPECIAL_INSTRUCTION,
    insertion_marker=INSERTION_MARKER,
    del_s_token_id=DELETION_TOKEN_IDS[0],
    del_a_token_id=DELETION_TOKEN_IDS[1]
)

In [8]:
dataset.cleanup_cache_files()

{'train': 0}

In [9]:
tokenized_dataset = dataset.map(mapper, batched=False, load_from_cache_file=False)
tokenized_dataset = tokenized_dataset["train"]
columns_to_remove = [
    "input", "correct_response", "incorrect_response", 
    "additional_info", "errors", "hallucinated_text"
]

tokenized_dataset = tokenized_dataset.remove_columns(columns_to_remove)


Map:   0%|          | 0/35019 [00:00<?, ? examples/s]

In [10]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 35019
})

In [11]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
print(split_dataset)
# train_dataset = split_dataset['train']
# eval_dataset = split_dataset['test']

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
        num_rows: 31517
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
        num_rows: 3502
    })
})


In [12]:
output_dir = "../../dataset/s1_training"
split_dataset.save_to_disk(output_dir)

Saving the dataset (0/1 shards):   0%|          | 0/31517 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3502 [00:00<?, ? examples/s]

In [16]:
dataset = datasets.load_from_disk("../../dataset/s1_training")
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 31517
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 3502
})


In [14]:
del_s_counter = 0
del_a_counter = 0

for i in range(len(train_dataset)):
    sample = train_dataset[i]
    for label in sample["hallucination_labels"]:
        if label == 1:
            del_s_counter += 1
            break
        elif label == 2:
            del_a_counter += 1
            break

print(del_s_counter, del_a_counter)
    

20129 0


In [15]:
del_s_counter = 0
del_a_counter = 0

for i in range(len(eval_dataset)):
    sample = eval_dataset[i]
    for label in sample["hallucination_labels"]:
        if label == 1:
            del_s_counter += 1
            break
        elif label == 2:
            del_a_counter += 1
            break

print(del_s_counter, del_a_counter)
    

2256 0


In [16]:
1/0

ZeroDivisionError: division by zero

In [16]:
max_len = 0
max_len_id = 0
for i in range(0, len(train_dataset)):
    sample = train_dataset[i]
    if len(sample["input_ids"]) > max_len:
        max_len = len(sample["input_ids"])
        max_len_id = i

print(max_len)
print(max_len_id)


avg_len = 0
for i in range(0, len(train_dataset)):
    sample = train_dataset[i]
    avg_len += len(sample["input_ids"])

print(avg_len / len(train_dataset))

1300
3397
350.61147234678623


In [ ]:
# for i in range(4000, 4600):
#     print(len(train_dataset[i]["input_ids"]))

In [ ]:
# 1/0

In [ ]:
# import torch

In [ ]:
# sample = train_dataset[5]
# sample["input_ids"] = torch.tensor(sample["input_ids"]).reshape(1, -1)
# sample["hallucination_labels"] = torch.tensor(sample["hallucination_labels"]).reshape(1, -1)
# sample["labels"] = torch.tensor(sample["labels"]).reshape(1, -1)

In [ ]:
# sample

In [ ]:
# print(sample["labels"][sample["labels"] != -100])
# print(sample["hallucination_labels"][sample["hallucination_labels"] != -100])

In [ ]:
# model.forward(
#     input_ids = sample["input_ids"],
#     hallucination_labels = sample["hallucination_labels"],
#     labels = sample["labels"]
# )

In [ ]:
# error_counter = 0
# for sample in eval_dataset:
#     hallucination_labels = [sample["hallucination_labels"][i] for i in range(len(sample["hallucination_labels"])) if sample["hallucination_labels"][i] != -100]
#     for label in hallucination_labels:
#         if label != 0:
#             error_counter += 1
#             break

# print(error_counter)


In [14]:
for i in range(100):
    sample = train_dataset[i]
    print(tokenizer.decode(sample["input_ids"]), "\n")
    tmp_hall_labels = [e for e in sample["hallucination_labels"] if e != -100]

    hall_text_idx = [i for i, label in enumerate(sample["hallucination_labels"]) if label == 1]
    hall_text = [sample["input_ids"][i] for i in hall_text_idx]

    print("--------------------------------\n")
    if hall_text_idx:
        print(tokenizer.decode(hall_text))
    else:
        print(tmp_hall_labels)
    
    # labels = [sample["labels"][i] for i in range(len(sample["labels"])) if sample["labels"][i] != -100]
    print(sample["labels"][-len(tmp_hall_labels):])
        
    
    print("########################################################\n\n")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a meticulous AI mathematician. Your task is to solve the following math problem.

Follow these steps carefully:
1. **Analyze the problem:** First, understand the given information and what is being asked.
2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.
3. **Solve or Explain:**
   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.
   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.

Your entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.
If you realize you have made a mistake, you must use one of the following correction tokens to fix it:

In [ ]:
# del_tokens = ["<DEL_S>", "<DEL_A>"]


# for i in range(100):
#     sample = train_dataset[i]
#     print(tokenizer.decode(sample["input_ids"]), "\n")
#     tmp_hall_labels = [e for e in sample["hallucination_labels"] if e != -100]

#     for j in range(1, 3):
#         hall_text_idx = [i for i, label in enumerate(sample["hallucination_labels"]) if label == j]
#         hall_text = [sample["input_ids"][i] for i in hall_text_idx]

#         print("--------------------------------\n")
#         if hall_text_idx:
#             print(f"Deletion token: {del_tokens[j-1]}")
#             print(tokenizer.decode(hall_text))
#         else:
#             print(tmp_hall_labels)
    
#     # labels = [sample["labels"][i] for i in range(len(sample["labels"])) if sample["labels"][i] != -100]
#     print(sample["labels"][-len(tmp_hall_labels):])
        
    
#     print("########################################################\n\n")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a meticulous AI mathematician. Your task is to solve the following math problem.

Follow these steps carefully:
1. **Analyze the problem:** First, understand the given information and what is being asked.
2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.
3. **Solve or Explain:**
   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.
   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.

Your entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.
If you realize you have made a mistake, you must use one of the following tools to correct it: Use <DE

In [ ]:
1/0

ZeroDivisionError: division by zero

In [ ]:
# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     TrainingArguments,
#     BitsAndBytesConfig,
# )
# import torch
# from src.trainer import SelfCorrectionTrainer

In [ ]:
# trainer = SelfCorrectionTrainer(model = "None")

In [ ]:
# sample = train_dataset[0]
# sample["input_ids"] = torch.tensor(sample["input_ids"]).reshape(1, -1)
# sample["hallucination_labels"] = torch.tensor(sample["hallucination_labels"]).reshape(1, -1)
# sample["labels"] = torch.tensor(sample["labels"]).reshape(1, -1)
# print(sample)

In [ ]:
# output = sample.copy()
# # output["logits"] = torch.tensor(output["input_ids"]).reshape(1, -1, 1)
# output["logits"] = torch.zeros(1, output["input_ids"].shape[-1], len(tokenizer.get_vocab())) - 100
# for i in range(output["input_ids"].shape[-1]):
#     output["logits"][0, i, output["input_ids"][0, i]] = 100
# output["hallucination_logits"] = torch.tensor(output["hallucination_labels"]).reshape(1, -1, 1)

In [ ]:
# output["logits"][0, :-1, ...] = output["logits"].clone()[0, 1:, ...]

In [ ]:
# trainer.compute_loss(sample, output, len(tokenizer.get_vocab()))

In [ ]:
# import torch

# logits = torch.zeros(1, 10, 100)
# hallucination_logits = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]).reshape(1, 10, 1)

# additional_logits = torch.zeros_like(logits)
# additional_logits[:, :, -3:] = hallucination_logits
# logits = logits + additional_logits
# print(logits)

In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# base_model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.2-1B-Instruct",
# )

In [ ]:
# print(base_model.lm_head.weight.shape)
# print(base_model.lm_head.weight[-5:, :10])

In [ ]:
# base_model.resize_token_embeddings(len(tokenizer.get_vocab())+3)

In [ ]:
# print(base_model.lm_head.weight.shape)
# print(base_model.lm_head.weight[-8:, :10])

In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import os
# import sys
# import torch

# # Add the project root directory to the Python path
# project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
# if project_root not in sys.path:
#     sys.path.insert(0, project_root)

In [ ]:
# path = "../../self-corrective-llama_untrained"
# # path = "meta-llama/Llama-3.2-1B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(path)
# tokenizer.pad_token = tokenizer.eos_token
# model = AutoModelForCausalLM.from_pretrained(path, trust_remote_code=True)

In [ ]:
model

SelfCorrectiveLlama(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_em

In [ ]:
import torch

def generate_with_hallucination_analysis(model, tokenizer, prompt_text, max_new_tokens=50):
    """
    Generates text and then runs a second pass to get hallucination logits
    for the generated tokens, avoiding interference with the generate loop.
    """
    # Ensure the model is in evaluation mode
    model.eval()
    
    # --- Pass 1: Generate the text ---
    # We use the standard, unmodified generate method.
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    
    # Decode the full generated text
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    
    # --- Pass 2: Run a single forward pass on the full sequence to get all logits ---
    # This is more efficient than collecting them step-by-step.
    with torch.no_grad():
        full_sequence_outputs = model(generated_ids)
    
    # Extract the hallucination logits from the output
    hallucination_logits = full_sequence_outputs.hallucination_logits
    
    # We only care about the logits for the *newly generated* tokens.
    # The logit at position `i` is the prediction for the token at `i+1`.
    prompt_len = inputs.input_ids.shape[1]
    generated_hallucination_logits = hallucination_logits[:, prompt_len-1:-1, :]
    
    return generated_text, generated_hallucination_logits

# --- Example Usage ---
text, hall_logits = generate_with_hallucination_analysis(
    model, 
    tokenizer, 
    "What is the capital of France?",
    max_new_tokens=50
)
print("Generated Text:", text)
print("Hallucination Logits Shape:", hall_logits.shape)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


deletion_logits:  tensor([[[-0.2549, -0.4001, -0.0107],
         [ 0.1313,  0.2042, -0.4412],
         [ 0.2938,  0.5907,  0.2403],
         [ 0.0054,  0.2033,  0.0871],
         [ 0.0671, -0.1945, -0.0231],
         [ 0.4582, -0.0522,  0.0768],
         [ 0.0837,  0.7528,  0.1571],
         [ 0.2778,  0.0508, -0.4632]]])
deletion_tokens_boost:  tensor([[[0.5738, 0.5130, 0.6878],
         [0.7610, 0.8005, 0.4967],
         [0.8508, 1.0315, 0.8205],
         [0.6959, 0.7999, 0.7377],
         [0.7272, 0.6006, 0.6817],
         [0.9482, 0.6674, 0.7323],
         [0.7359, 1.1387, 0.7748],
         [0.8417, 0.7189, 0.4881]]])
deletion_logits:  tensor([[[-0.1823, -0.2212, -0.0581]]])
deletion_tokens_boost:  tensor([[[0.6062, 0.5887, 0.6645]]])
deletion_logits:  tensor([[[-0.3116,  0.0493, -0.3290]]])
deletion_tokens_boost:  tensor([[[0.5494, 0.7181, 0.5421]]])
deletion_logits:  tensor([[[0.0664, 0.0207, 0.0339]]])
deletion_tokens_boost:  tensor([[[0.7269, 0.7036, 0.7103]]])
deletion_logits:

In [ ]:
hall_logits

tensor([[[ 0.1624,  0.2778,  0.0508, -0.4632],
         [ 0.2900, -0.1823, -0.2212, -0.0581],
         [ 0.5709, -0.3116,  0.0493, -0.3290],
         [ 0.3711,  0.0664,  0.0207,  0.0339],
         [ 0.2748,  0.1820, -0.1783, -0.2430],
         [ 0.0483,  0.5533, -0.1029,  0.2045],
         [ 0.1496,  0.2198,  0.5276, -0.2073],
         [ 0.2203,  0.8987,  0.0864, -0.2970],
         [ 0.4481,  0.2962, -0.0133, -0.4136],
         [ 0.1487,  0.4633,  0.2440, -0.3378],
         [ 0.3457, -0.4774,  0.1886,  0.1360],
         [-0.0306,  0.1782,  0.6855, -0.0944],
         [ 0.2979, -0.0923,  0.2596, -0.0361],
         [-0.0962, -0.4460,  0.2367, -0.0086],
         [ 0.3327, -0.2294, -0.2962, -0.1661],
         [ 0.2646,  0.1721, -0.3685, -0.3746],
         [-0.3313,  0.5522,  0.2868,  0.2542],
         [-0.0789,  0.2417,  0.5364, -0.4203],
         [-0.0474, -0.1057,  0.5387, -0.2691],
         [-0.3274, -0.0347,  0.2571, -0.3841],
         [-0.1355, -0.1274,  0.3666, -0.2415],
         [ 0.

In [ ]:
p1 = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a specialized question-answering AI. Your task is to give a concise answer to the question using *only* the provided context. Make sure to always give an answer. Use <DEL_W>, <DEL_S> or <DEL_A> tokens if needed.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n'''\nSichuan has been historically known as the \"Province of Abundance\". It is one of the major agricultural production bases of China. Grain, including rice and wheat, is the major product with output that ranked first in China in 1999. Commercial crops include citrus fruits, sugar cane, sweet potatoes, peaches and grapes. Sichuan also had the largest output of pork among all the provinces and the second largest output of silkworm cocoons in 1999. Sichuan is rich in mineral resources. It has more than 132 kinds of proven underground mineral resources including vanadium, titanium, and lithium being the largest in China. The Panxi region alone possesses 13.3% of the reserves of iron, 93% of titanium, 69% of vanadium, and 83% of the cobalt of the whole country. Sichuan also possesses China's largest proven natural gas reserves, the majority of which is transported to more developed eastern regions.\n'''\n\nQuestion: What are the major agricultural outputs of Sichuan?<|eot_id|><|start_header_id|>assistant<|end_header_id|>",
# p1 = "What is the capital of France?"
inputs = tokenizer(p1, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[128000, 128000, 128006,   9125, 128007,    271,   2675,    527,    264,
          28175,   3488,     12,    598,     86,   4776,  15592,     13,   4718,
           3465,    374,    311,   3041,    264,  64694,   4320,    311,    279,
           3488,   1701,    353,   3323,      9,    279,   3984,   2317,     13,
           7557,   2771,    311,   2744,   3041,    459,   4320,     13,   5560,
            220, 128256,     11,    220, 128257,    477,    220, 128258,  11460,
            422,   4460,     13, 128009, 128006,    882, 128007,    271,   2014,
            512,  15029,     50,    718,  10602,    706,   1027,  35901,   3967,
            439,    279,    330,  52174,    315,   3765,  98681,   3343,   1102,
            374,    832,    315,    279,   3682,  29149,   5788,  23963,    315,
           5734,     13,  75374,     11,   2737,  20228,    323,  34153,     11,
            374,    279,   3682,   2027,    449,   2612,    430,  21682,   1176,
            30

In [ ]:
# res = model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], output_hallucination_logits=True)
res = model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


deletion_logits:  tensor([[[-0.2549, -0.4001, -0.0107],
         [-0.2549, -0.4001, -0.0107],
         [-0.0740, -0.7530, -0.1258],
         [ 0.3389, -0.2641, -0.0171],
         [ 0.1632, -0.4458, -0.5068],
         [-0.5948, -0.2511,  0.5737],
         [-0.3839,  0.2306,  0.0429],
         [ 0.1176,  0.1183, -0.5446],
         [ 0.2556,  0.5864, -0.2548],
         [ 0.5588,  0.5449, -0.2947],
         [ 0.4177,  0.5752,  0.1347],
         [ 0.3763, -0.2367,  0.0129],
         [-0.5349,  0.7762, -0.6208],
         [ 0.1676,  0.5535, -0.0292],
         [ 0.1122,  0.5834,  0.2341],
         [-0.0277,  0.0532,  0.7694],
         [-0.1124, -0.2601, -0.1624],
         [ 0.5338,  0.2463,  0.5683],
         [ 0.3394,  0.2396,  0.5887],
         [ 0.4744,  0.6960,  0.3231],
         [ 0.4138,  0.6392,  0.6392],
         [-0.3610,  0.1510,  0.6224],
         [ 0.1828, -0.0884,  0.8144],
         [ 0.3716, -0.1513, -0.0740],
         [ 0.0584,  0.8696,  0.2812],
         [-0.0152,  0.6125,  0.7

In [ ]:
model.original_vocab_size

128256

In [ ]:
res

tensor([[128000, 128000, 128006,   9125, 128007,    271,   2675,    527,    264,
          28175,   3488,     12,    598,     86,   4776,  15592,     13,   4718,
           3465,    374,    311,   3041,    264,  64694,   4320,    311,    279,
           3488,   1701,    353,   3323,      9,    279,   3984,   2317,     13,
           7557,   2771,    311,   2744,   3041,    459,   4320,     13,   5560,
            220, 128256,     11,    220, 128257,    477,    220, 128258,  11460,
            422,   4460,     13, 128009, 128006,    882, 128007,    271,   2014,
            512,  15029,     50,    718,  10602,    706,   1027,  35901,   3967,
            439,    279,    330,  52174,    315,   3765,  98681,   3343,   1102,
            374,    832,    315,    279,   3682,  29149,   5788,  23963,    315,
           5734,     13,  75374,     11,   2737,  20228,    323,  34153,     11,
            374,    279,   3682,   2027,    449,   2612,    430,  21682,   1176,
            304,   5734,    

In [ ]:
tokens = res

In [ ]:
# tokens, hall = res

In [ ]:
tokenizer.decode(tokens[0])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a specialized question-answering AI. Your task is to give a concise answer to the question using *only* the provided context. Make sure to always give an answer. Use <DEL_W>, <DEL_S> or <DEL_A> tokens if needed.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n\'\'\'\nSichuan has been historically known as the "Province of Abundance". It is one of the major agricultural production bases of China. Grain, including rice and wheat, is the major product with output that ranked first in China in 1999. Commercial crops include citrus fruits, sugar cane, sweet potatoes, peaches and grapes. Sichuan also had the largest output of pork among all the provinces and the second largest output of silkworm cocoons in 1999. Sichuan is rich in mineral resources. It has more than 132 kinds of proven underground mineral resources including vanadium, titanium, and lithium being the largest in China. The P

In [ ]:
1/0

ZeroDivisionError: division by zero

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model_name = "meta-llama/Llama-3.2-1B-Instruct"
special_tokens = ["<DEL_W>", "<DEL_S>", "<DEL_A>"]
intermediate_save_path = "./temp_resized_model"

# --- Step 1: Load, Resize, and Save the STANDARD Llama model ---
print("--- Step 1: Loading and resizing original LlamaForCausalLM ---")
# Load the original, standard model class
original_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
original_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
)

# Add tokens and resize the embeddings of the standard model
original_tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
original_model.resize_token_embeddings(len(original_tokenizer))

# Save this intermediate model. It is still a LlamaForCausalLM, just with a bigger lm_head
print(f"--- Saving resized LlamaForCausalLM to {intermediate_save_path} ---")
original_model.save_pretrained(intermediate_save_path)
original_tokenizer.save_pretrained(intermediate_save_path)

# Clear memory
del original_model
del original_tokenizer
torch.cuda.empty_cache()

--- Step 1: Loading and resizing original LlamaForCausalLM ---


KeyboardInterrupt: 

In [ ]:
# --- Step 2: Load the RESIZED model and run the prompt ---
print("\n--- Step 2: Loading the RESIZED LlamaForCausalLM and testing ---")
# Load the model you just saved. This is still the standard class.
resized_tokenizer = AutoTokenizer.from_pretrained(intermediate_save_path)
resized_model = AutoModelForCausalLM.from_pretrained(
    intermediate_save_path,
)

print(resized_model.lm_head.weight.shape)

prompt = "What is the capital of France?"
inputs = resized_tokenizer(prompt, return_tensors="pt")
outputs = resized_model.generate(**inputs, max_new_tokens=50)
print("--- Output from RESIZED LlamaForCausalLM ---")
print(resized_tokenizer.decode(outputs[0], skip_special_tokens=False))

# Clear memory
del resized_model
del resized_tokenizer
torch.cuda.empty_cache()


--- Step 2: Loading the RESIZED LlamaForCausalLM and testing ---


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


torch.Size([128259, 2048])
--- Output from RESIZED LlamaForCausalLM ---
<|begin_of_text|>What is the capital of France? Paris
The capital of France is Paris, which is also the most populous city in the country. Paris is known for its rich history, art, fashion, and cuisine. It is home to many famous landmarks, such as the Eiffel Tower


In [ ]:
# # --- Step 3 (Optional): Confirm your custom model behaves the same ---
# print("\n--- Step 3: Loading the RESIZED model into your custom class ---")
# # This is the final check. We load the same resized model, but now we
# # let it be interpreted as your custom class.
# # We must include the local modeling.py file via `trust_remote_code=True`.
# custom_tokenizer = AutoTokenizer.from_pretrained(intermediate_save_path)
# custom_model = AutoModelForCausalLM.from_pretrained(
#     intermediate_save_path,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
#     trust_remote_code=True # This will use your local `modeling.py`
# )

# inputs = custom_tokenizer(prompt, return_tensors="pt").to("cuda")
# outputs = custom_model.generate(**inputs, max_new_tokens=50)
# print("--- Output from RESIZED SelfCorrectiveLlama ---")
# print(custom_tokenizer.decode(outputs[0], skip_special_tokens=False))